In [ ]:
#DATASET LOAD
# Load all hospital datasets from the raw data folders
# and store them as Pandas DataFrames for analysis.

import pandas as pd
from pathlib import Path

base = Path("../data/raw")

admissions = pd.read_csv(base / "admissions/admissions.csv")
appointments = pd.read_csv(base / "appointments/appointments.csv")
beds = pd.read_csv(base / "beds/beds.csv")
departments = pd.read_csv(base / "departments/departments.csv")
equipment = pd.read_csv(base / "equipment/equipment.csv")

print("Files loaded successfully!")

Files loaded successfully!


In [ ]:
# Check the number of rows, columns, and column names
# in each hospital dataset.
datasets = {
    "Admissions": admissions,
    "Appointments": appointments,
    "Beds": beds,
    "Departments": departments,
    "Equipment": equipment
}

for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Column names:")
    print(df.columns.tolist())


Admissions
Rows: 50000
Columns: 11
Column names:
['admission_id', 'patient_id', 'department_id', 'admission_date', 'discharge_date', 'admission_type', 'age', 'gender', 'length_of_stay', 'bed_type', 'discharge_status']

Appointments
Rows: 60000
Columns: 8
Column names:
['appointment_id', 'patient_id', 'department_id', 'appointment_date', 'appointment_type', 'booking_date', 'wait_time_days', 'appointment_status']

Beds
Rows: 15344
Columns: 8
Column names:
['bed_record_id', 'department_id', 'record_date', 'bed_type', 'total_beds', 'occupied_beds', 'available_beds', 'occupancy_rate']

Departments
Rows: 14
Columns: 4
Column names:
['department_id', 'department_name', 'department_type', 'hospital_area']

Equipment
Rows: 5000
Columns: 10
Column names:
['equipment_id', 'department_id', 'equipment_type', 'installation_date', 'last_maintenance_date', 'next_maintenance_date', 'equipment_status', 'usage_hours', 'maintenance_count', 'failure_count']


In [ ]:
# Check each dataset for missing values
# and identify which columns contain missing data.

for name, df in datasets.items():
    print("\n==============================")
    print(name)
    print("==============================")
    print(df.isnull().sum())


Admissions
admission_id           0
patient_id             0
department_id          0
admission_date         0
discharge_date         0
admission_type         0
age                  750
gender               500
length_of_stay         0
bed_type             400
discharge_status    1000
dtype: int64

Appointments
appointment_id          0
patient_id              0
department_id           0
appointment_date        0
appointment_type      600
booking_date            0
wait_time_days        300
appointment_status    900
dtype: int64

Beds
bed_record_id     0
department_id     0
record_date       0
bed_type          0
total_beds        0
occupied_beds     0
available_beds    0
occupancy_rate    0
dtype: int64

Departments
department_id      0
department_name    0
department_type    0
hospital_area      0
dtype: int64

Equipment
equipment_id              0
department_id             0
equipment_type            0
installation_date         0
last_maintenance_date    50
next_maintenance_date    

In [4]:
# Check how many completely duplicate rows exist in each dataset
for name, df in datasets.items():
    print("\n==============================")
    print(name)
    print("==============================")
    print("Duplicate rows:", df.duplicated().sum())


Admissions
Duplicate rows: 0

Appointments
Duplicate rows: 0

Beds
Duplicate rows: 0

Departments
Duplicate rows: 0

Equipment
Duplicate rows: 0


In [5]:
# Check the data type of every column in each dataset
# to identify columns that may need type conversion.

for name, df in datasets.items():
    print("\n==============================")
    print(name)
    print("==============================")
    print(df.dtypes)


Admissions
admission_id         object
patient_id           object
department_id        object
admission_date       object
discharge_date       object
admission_type       object
age                 float64
gender               object
length_of_stay        int64
bed_type             object
discharge_status     object
dtype: object

Appointments
appointment_id         object
patient_id             object
department_id          object
appointment_date       object
appointment_type       object
booking_date           object
wait_time_days        float64
appointment_status     object
dtype: object

Beds
bed_record_id      object
department_id      object
record_date        object
bed_type           object
total_beds          int64
occupied_beds       int64
available_beds      int64
occupancy_rate    float64
dtype: object

Departments
department_id      object
department_name    object
department_type    object
hospital_area      object
dtype: object

Equipment
equipment_id             obj

In [6]:
# Check the minimum and maximum dates in each dataset
# to verify the date ranges and identify invalid date values.
# Convert date columns to datetime format and display their date ranges

date_columns = {
    "Admissions": ["admission_date", "discharge_date"],
    "Appointments": ["appointment_date", "booking_date"],
    "Beds": ["record_date"],
    "Equipment": ["installation_date", "last_maintenance_date", "next_maintenance_date"]
}

for name, columns in date_columns.items():
    print("\n==============================")
    print(name)
    print("==============================")
    
    for col in columns:
        datasets[name][col] = pd.to_datetime(datasets[name][col], errors="coerce")
        print(col, ":", datasets[name][col].min(), "to", datasets[name][col].max())


Admissions
admission_date : 2023-01-01 00:00:00 to 2025-12-31 00:00:00
discharge_date : 2023-01-01 00:00:00 to 2026-01-27 00:00:00

Appointments
appointment_date : 2023-01-01 00:00:00 to 2025-12-31 00:00:00
booking_date : 2022-11-04 00:00:00 to 2025-12-29 00:00:00

Beds
record_date : 2023-01-01 00:00:00 to 2025-12-31 00:00:00

Equipment
installation_date : 2013-01-04 00:00:00 to 2025-12-29 00:00:00
last_maintenance_date : 2013-06-09 00:00:00 to 2025-12-31 00:00:00
next_maintenance_date : 2013-08-10 00:00:00 to 2026-06-27 00:00:00


In [7]:
# Check important business rules and relationships between hospital datasets
# to find invalid department IDs, patient IDs, dates, lengths of stay,
# bed calculations, occupancy rates, and equipment maintenance dates.

print("Admissions: discharge before admission")
print((admissions["discharge_date"] < admissions["admission_date"]).sum())

print("\nAdmissions: length of stay mismatch")
expected_los = (admissions["discharge_date"] - admissions["admission_date"]).dt.days
print((admissions["length_of_stay"] != expected_los).sum())

print("\nAdmissions: invalid department IDs")
print((~admissions["department_id"].isin(departments["department_id"])).sum())

print("\nAppointments: invalid department IDs")
print((~appointments["department_id"].isin(departments["department_id"])).sum())

print("\nAppointments: booking after appointment")
print((appointments["booking_date"] > appointments["appointment_date"]).sum())

print("\nBeds: occupied greater than total")
print((beds["occupied_beds"] > beds["total_beds"]).sum())

print("\nBeds: available beds mismatch")
expected_available = beds["total_beds"] - beds["occupied_beds"]
print((beds["available_beds"] != expected_available).sum())

print("\nBeds: occupancy rate mismatch")
expected_rate = beds["occupied_beds"] / beds["total_beds"]
print((abs(beds["occupancy_rate"] - expected_rate) > 0.01).sum())

print("\nEquipment: invalid maintenance dates")
print((equipment["last_maintenance_date"] < equipment["installation_date"]).sum())

print("\nEquipment: next maintenance before last maintenance")
print((equipment["next_maintenance_date"] < equipment["last_maintenance_date"]).sum())

Admissions: discharge before admission
0

Admissions: length of stay mismatch
0

Admissions: invalid department IDs
0

Appointments: invalid department IDs
0

Appointments: booking after appointment
0

Beds: occupied greater than total
0

Beds: available beds mismatch
0

Beds: occupancy rate mismatch
15344

Equipment: invalid maintenance dates
0

Equipment: next maintenance before last maintenance
0


In [8]:
# Check whether patient IDs in appointments also exist in admissions
# to verify that the two datasets can be reliably connected.
appointment_patients = set(appointments["patient_id"])
admission_patients = set(admissions["patient_id"])

missing_patients = appointment_patients - admission_patients

print("Patients in appointments:", len(appointment_patients))
print("Patients in admissions:", len(admission_patients))
print("Appointment patients not found in admissions:", len(missing_patients))

Patients in appointments: 22285
Patients in admissions: 20781
Appointment patients not found in admissions: 5991


In [9]:

# Check how many patients appear in both admissions and appointments
# to measure the overlap between the two datasets.

common_patients = admission_patients & appointment_patients

print("Patients in both datasets:", len(common_patients))
print("Percentage of admission patients with appointments:",
      round(len(common_patients) / len(admission_patients) * 100, 2), "%")

Patients in both datasets: 16294
Percentage of admission patients with appointments: 78.41 %


In [10]:
# Check whether every department ID used in hospital datasets
# exists in the master departments dataset.
department_ids = set(departments["department_id"])

print("Admissions invalid departments:",
      (~admissions["department_id"].isin(department_ids)).sum())

print("Appointments invalid departments:",
      (~appointments["department_id"].isin(department_ids)).sum())

print("Beds invalid departments:",
      (~beds["department_id"].isin(department_ids)).sum())

print("Equipment invalid departments:",
      (~equipment["department_id"].isin(department_ids)).sum())

Admissions invalid departments: 0
Appointments invalid departments: 0
Beds invalid departments: 0
Equipment invalid departments: 0


In [11]:
# Calculate the number and percentage of missing values
# in each column to decide how each missing value should be handled.
for name, df in datasets.items():
    print("\n==============================")
    print(name)
    print("==============================")

    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    missing = pd.DataFrame({
        "Missing Count": missing_count,
        "Missing %": missing_percent.round(2)
    })

    print(missing[missing["Missing Count"] > 0])


Admissions
                  Missing Count  Missing %
age                         750        1.5
gender                      500        1.0
bed_type                    400        0.8
discharge_status           1000        2.0

Appointments
                    Missing Count  Missing %
appointment_type              600        1.0
wait_time_days                300        0.5
appointment_status            900        1.5

Beds
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []

Departments
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []

Equipment
                       Missing Count  Missing %
last_maintenance_date             50        1.0
next_maintenance_date             75        1.5


In [12]:
# Check the most common values in columns with missing data
# so we can choose appropriate values for categorical columns.
print("Admissions")
print("Gender:\n", admissions["gender"].value_counts(dropna=False))
print("\nBed Type:\n", admissions["bed_type"].value_counts(dropna=False))
print("\nDischarge Status:\n", admissions["discharge_status"].value_counts(dropna=False))

print("\nAppointments")
print("Appointment Type:\n", appointments["appointment_type"].value_counts(dropna=False))
print("\nAppointment Status:\n", appointments["appointment_status"].value_counts(dropna=False))

Admissions
Gender:
 gender
Female    26291
Male      21862
Other      1347
NaN         500
Name: count, dtype: int64

Bed Type:
 bed_type
General         23257
Semi-Private     9579
ICU              9467
Private          7297
NaN               400
Name: count, dtype: int64

Discharge Status:
 discharge_status
Discharged     41007
Transferred     3545
Referred        3178
Deceased        1270
NaN             1000
Name: count, dtype: int64

Appointments
Appointment Type:
 appointment_type
Follow-up           22592
New Consultation    18897
Diagnostic           9997
Procedure            7914
NaN                   600
Name: count, dtype: int64

Appointment Status:
 appointment_status
Completed      47460
Cancelled       4577
No-Show         4308
Rescheduled     2755
NaN              900
Name: count, dtype: int64


In [13]:
# Check the median values of numeric columns with missing data
# so we can use statistically appropriate values during cleaning.
print("Median age:", admissions["age"].median())
print("Median wait time:", appointments["wait_time_days"].median())

Median age: 46.0
Median wait time: 8.0


In [14]:
# Create copies of the raw datasets and handle missing values
# using appropriate methods without modifying the original raw files.
# Create copies so the original raw data remains unchanged
admissions_clean = admissions.copy()
appointments_clean = appointments.copy()
beds_clean = beds.copy()
departments_clean = departments.copy()
equipment_clean = equipment.copy()

# Fill missing numeric values with the median
admissions_clean["age"] = admissions_clean["age"].fillna(46.0)
appointments_clean["wait_time_days"] = appointments_clean["wait_time_days"].fillna(8.0)

# Fill missing categorical values with "Unknown"
admissions_clean["gender"] = admissions_clean["gender"].fillna("Unknown")
admissions_clean["bed_type"] = admissions_clean["bed_type"].fillna("Unknown")
admissions_clean["discharge_status"] = admissions_clean["discharge_status"].fillna("Unknown")

appointments_clean["appointment_type"] = appointments_clean["appointment_type"].fillna("Unknown")
appointments_clean["appointment_status"] = appointments_clean["appointment_status"].fillna("Unknown")

print("Basic missing-value cleaning completed.")

Basic missing-value cleaning completed.


In [15]:
# Check whether missing maintenance dates occur together
# so we can choose a safe treatment without inventing information.
print("Missing last maintenance:",
      equipment_clean["last_maintenance_date"].isna().sum())

print("Missing next maintenance:",
      equipment_clean["next_maintenance_date"].isna().sum())

print("Both dates missing:",
      (equipment_clean["last_maintenance_date"].isna() &
       equipment_clean["next_maintenance_date"].isna()).sum())

Missing last maintenance: 50
Missing next maintenance: 75
Both dates missing: 0


In [16]:
# Keep missing equipment maintenance dates as missing values
# because inventing maintenance dates could create false operational information.

equipment_clean["last_maintenance_date"] = pd.to_datetime(
    equipment_clean["last_maintenance_date"],
    errors="coerce"
)

equipment_clean["next_maintenance_date"] = pd.to_datetime(
    equipment_clean["next_maintenance_date"],
    errors="coerce"
)

print("Equipment maintenance dates handled.")


Equipment maintenance dates handled.


In [17]:
# Verify that the cleaning process handled the intended missing values
# and identify any remaining missing values that were intentionally preserved.
clean_datasets = {
    "Admissions": admissions_clean,
    "Appointments": appointments_clean,
    "Beds": beds_clean,
    "Departments": departments_clean,
    "Equipment": equipment_clean
}

for name, df in clean_datasets.items():
    print("\n==============================")
    print(name)
    print("==============================")
    print(df.isnull().sum()[df.isnull().sum() > 0])


Admissions
Series([], dtype: int64)

Appointments
Series([], dtype: int64)

Beds
Series([], dtype: int64)

Departments
Series([], dtype: int64)

Equipment
last_maintenance_date    50
next_maintenance_date    75
dtype: int64


In [18]:
# Save the cleaned datasets into data/processed
# while keeping the original raw datasets unchanged.
from pathlib import Path

processed_base = Path("../data/processed")

# Create folders for each processed dataset
for folder in ["admissions", "appointments", "beds", "departments", "equipment"]:
    (processed_base / folder).mkdir(parents=True, exist_ok=True)

# Save cleaned datasets
admissions_clean.to_csv(processed_base / "admissions/admissions_clean.csv", index=False)
appointments_clean.to_csv(processed_base / "appointments/appointments_clean.csv", index=False)
beds_clean.to_csv(processed_base / "beds/beds_clean.csv", index=False)
departments_clean.to_csv(processed_base / "departments/departments_clean.csv", index=False)
equipment_clean.to_csv(processed_base / "equipment/equipment_clean.csv", index=False)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!
